# 07 — Trip Duration Prediction Agent

**Type:** Side project  
**Purpose:** Train a trip-duration model on Silver taxi data **enriched with historical weather, congestion, and holiday features**, then wire it to an LLM agent that fetches live context and returns a duration estimate.

---

### Architecture

```
User: "I'm at Times Square, taxi to JFK please"
         │
         ▼
   LLM Agent (Llama 3.3 70B)
         │
   ┌─────┴──────────────────────────────────────┐
   │  Tool calls (function calling)            │
   │  1. get_datetime_features()                │  ← current NYC time + is_holiday
   │  2. geocode_address(pickup)                │  ← lat/lon via Nominatim
   │  3. geocode_address(dropoff)               │  ← lat/lon via Nominatim
   │  4. get_weather(lat, lon)                  │  ← Open-Meteo live forecast
   │  5. get_congestion(hour, day)              │  ← historical avg demand
   │  6. predict_duration(all features)         │  ← trained HistGBT model
   └────────────────────────────────────────────┘
         │
         ▼
   "Based on current conditions (3pm Thursday, light rain,
    high congestion), your trip should take ~42 min (~12.4 mi)."
```

### Training data enrichment

| Source | Features | Join key | Inference source |
|--------|----------|----------|------------------|
| **Silver taxi data** | location, distance, time, passengers, rate code | — | User query + geocoding |
| **Open-Meteo Archive** | temperature, precipitation, snowfall, wind speed | date + hour | Open-Meteo live forecast API |
| **Congestion proxy** | hourly trip count (taxi demand = traffic proxy) | date + hour | Historical avg for that hour/day |
| **Holiday calendar** | is_holiday flag | date | Python date lookup |

### Key design decisions

| Decision | Rationale |
|----------|-----------|
| **Duration, not wait time** | Dataset only has pickup→dropoff events, not order→pickup. |
| **Duration capped at 60 min** | P99.5 = 61 min. Capping retains 99.43% of data, removes long-tail outliers. |
| **Weather joined by date+hour** | Historical weather at Central Park — city-wide, matches all trips in that hour. |
| **Congestion = taxi trip count** | No traffic API for 2015-2016; taxi volume is a strong congestion proxy. |
| **Holidays** | 5 US federal holidays in dataset range; dramatically change traffic patterns. |
| **Haversine × 1.3 for distance** | At inference we only have coordinates; road distance ≈ straight-line × 1.3 for NYC. |

## Setup

In [0]:
%pip install mlflow scikit-learn requests pytz langchain langgraph databricks-langchain --quiet

In [0]:
%restart_python

In [0]:
import importlib
import math

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import src.constants
import src.transforms
from databricks_langchain import ChatDatabricks
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda
from langgraph.graph import END, StateGraph
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from pyspark.sql import functions as F
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

importlib.reload(src.constants)
importlib.reload(src.transforms)

from src.constants import (  # noqa: E402
    AGENT_LLM_ENDPOINT,
    DURATION_CAP_MIN,
    DURATION_MODEL_NAME,
    HOLIDAYS,
    LAT_LON_BIN_SIZE,
    ML_DURATION_TARGET_COLUMN,
    ML_FEATURE_COLUMNS,
    ML_RANDOM_STATE,
    ML_SAMPLE_FRACTION,
    ML_TEST_SIZE,
    SILVER_TABLE,
    WEATHER_TABLE,
)

# LangGraph + LangChain autolog gives rich MLflow traces
# (every agent step + tool call appears as a separate span)
mlflow.langchain.autolog()

# LLM via Databricks Foundation Model API
llm = ChatDatabricks(endpoint=AGENT_LLM_ENDPOINT)

print("Setup complete")
print(f"  Duration target : {ML_DURATION_TARGET_COLUMN}")
print(f"  Duration cap    : {DURATION_CAP_MIN} min")
print(f"  Weather table   : {WEATHER_TABLE}")
print(f"  Model name      : {DURATION_MODEL_NAME}")
print(f"  LLM endpoint    : {AGENT_LLM_ENDPOINT}")

## Enriched Feature Table

Joins Silver taxi data with:
1. **Weather** (Open-Meteo Archive, via Delta table from `07b_weather_history.ipynb`) by date + hour
2. **Congestion proxy** (city-wide hourly taxi trip count, self-derived from Silver) by date + hour
3. **Holiday flag** (US federal holidays within date range)

Duration capped at 60 min (P99.5 = 61). This removes 0.57% of rows and eliminates the extreme outliers that were destroying RMSE in the baseline model.

In [0]:
# ── Read Silver & weather ─────────────────────────────────────────────────────
silver_df = spark.read.table(SILVER_TABLE)
weather_df = spark.read.table(WEATHER_TABLE)
print(f"Silver rows:  {silver_df.count():,}")
print(f"Weather rows: {weather_df.count():,}")

# ── Add date + hour columns for joins ────────────────────────────────────────
silver_df = silver_df.withColumn(
    "trip_date", F.to_date("tpep_pickup_datetime")
).withColumn("trip_hour", F.hour("tpep_pickup_datetime"))

# ── 1) Join weather by date + hour ───────────────────────────────────────────
feature_df = silver_df.join(
    weather_df,
    (silver_df.trip_date == F.to_date(weather_df.date))
    & (silver_df.trip_hour == weather_df.hour),
    "inner",
).drop(weather_df.date, weather_df.hour)

print(f"After weather join: {feature_df.count():,}")

# ── 2) Congestion proxy: city-wide trip count per (date, hour) ───────────────
congestion_df = silver_df.groupBy("trip_date", "trip_hour").agg(
    F.count("*").alias("hourly_trip_count")
)
feature_df = feature_df.join(
    congestion_df.withColumnRenamed("trip_date", "c_date").withColumnRenamed(
        "trip_hour", "c_hour"
    ),
    (feature_df.trip_date == F.col("c_date"))
    & (feature_df.trip_hour == F.col("c_hour")),
    "left",
).drop("c_date", "c_hour")

# ── 3) Holiday flag ──────────────────────────────────────────────────────────
holiday_list = list(HOLIDAYS)
feature_df = feature_df.withColumn(
    "is_holiday",
    F.when(F.col("trip_date").cast("string").isin(holiday_list), 1).otherwise(0),
)

# ── 4) Cap duration at 60 min ────────────────────────────────────────────────
feature_df = feature_df.filter(
    (F.col(ML_DURATION_TARGET_COLUMN) > 0)
    & (F.col(ML_DURATION_TARGET_COLUMN) <= DURATION_CAP_MIN)
)

# ── Select features + target ─────────────────────────────────────────────────
enriched_features = [
    *ML_FEATURE_COLUMNS,
    "temperature_f",
    "precipitation_inch",
    "snowfall_inch",
    "wind_speed_mph",
    "hourly_trip_count",
    "is_holiday",
]
feature_df = feature_df.select(*enriched_features, ML_DURATION_TARGET_COLUMN)

# Drop nulls (rate_code_id NULLs, empty zones, weather nulls)
feature_df = feature_df.dropna()
feature_df = feature_df.filter(
    (F.col("pickup_zone") != "") & (F.col("dropoff_zone") != "")
)

# ── Parse zone strings into numeric lat/lon bins ─────────────────────────────
for prefix in ["pickup", "dropoff"]:
    feature_df = feature_df.withColumn(
        f"{prefix}_lat_bin",
        F.split(F.col(f"{prefix}_zone"), ",")[0].cast("double"),
    ).withColumn(
        f"{prefix}_lon_bin",
        F.split(F.col(f"{prefix}_zone"), ",")[1].cast("double"),
    )
feature_df = feature_df.drop("pickup_zone", "dropoff_zone")

# Cast booleans → int for sklearn
feature_df = feature_df.withColumn("is_weekend", F.col("is_weekend").cast("int"))

# ── Sample ────────────────────────────────────────────────────────────────────
feature_df = feature_df.sample(fraction=ML_SAMPLE_FRACTION, seed=ML_RANDOM_STATE)
pdf = feature_df.toPandas()

# One-hot encode rate_code_id
pdf = pd.get_dummies(pdf, columns=["rate_code_id"], prefix="rc", dtype=int)

print(f"\nEnriched feature table: {pdf.shape}")
print(
    f"Target mean: {pdf[ML_DURATION_TARGET_COLUMN].mean():.1f} min "
    f"| std: {pdf[ML_DURATION_TARGET_COLUMN].std():.1f} min "
    f"| max: {pdf[ML_DURATION_TARGET_COLUMN].max():.0f} min"
)

X = pdf.drop(columns=[ML_DURATION_TARGET_COLUMN])
y = pdf[ML_DURATION_TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=ML_TEST_SIZE, random_state=ML_RANDOM_STATE
)
print(f"Train: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(
    "\nNew features: temperature_f, precipitation_inch, snowfall_inch, "
    "wind_speed_mph, hourly_trip_count, is_holiday"
)

# ── Build congestion lookup for inference ─────────────────────────────────────
# Average hourly trip count by (hour_of_day, day_of_week) across all dates
congestion_lookup = (
    congestion_df.withColumn("dow", F.dayofweek("trip_date"))
    .groupBy(F.col("trip_hour").alias("lk_hour"), F.col("dow").alias("lk_dow"))
    .agg(F.round(F.avg("hourly_trip_count")).cast("int").alias("avg_trips"))
    .toPandas()
)
congestion_lookup = dict(
    zip(
        zip(congestion_lookup["lk_hour"], congestion_lookup["lk_dow"]),
        congestion_lookup["avg_trips"],
    )
)
congestion_median = int(np.median(list(congestion_lookup.values())))
print(f"\nCongestion lookup: {len(congestion_lookup)} (hour, day_of_week) entries")
print(f"Median hourly trips: {congestion_median:,}")

# Average weather conditions (used as baseline for route-specific comparisons)
avg_weather = {
    "temperature_f": float(pdf["temperature_f"].mean()),
    "precipitation_inch": float(pdf["precipitation_inch"].mean()),
    "snowfall_inch": float(pdf["snowfall_inch"].mean()),
    "wind_speed_mph": float(pdf["wind_speed_mph"].mean()),
}
print(f"Avg weather: {avg_weather}")

## Duration Model — Train & Register

In [0]:
# ── Train HistGradientBoostingRegressor on enriched features ──────────────────
dur_params = {
    "max_iter": 300,
    "max_depth": 8,
    "learning_rate": 0.05,
    "min_samples_leaf": 50,
    "random_state": ML_RANDOM_STATE,
}

duration_model = HistGradientBoostingRegressor(**dur_params)
duration_model.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred = duration_model.predict(X_test)
dur_metrics = {
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "mae": float(mean_absolute_error(y_test, y_pred)),
    "r2": float(r2_score(y_test, y_pred)),
}

# ── Log & register to MLflow ──────────────────────────────────────────────────
input_example = X_train.iloc[:5]

with mlflow.start_run(run_name="duration_enriched_HistGBT"):
    mlflow.log_params(dur_params)
    mlflow.log_param("model_type", "HistGradientBoostingRegressor")
    mlflow.log_param("target", ML_DURATION_TARGET_COLUMN)
    mlflow.log_param("duration_cap_min", DURATION_CAP_MIN)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", X_train.shape[0])
    mlflow.log_param("test_rows", X_test.shape[0])
    mlflow.log_param("enrichment", "weather+congestion+holidays")
    mlflow.log_metrics(dur_metrics)
    mlflow.sklearn.log_model(
        duration_model,
        name="model",
        input_example=input_example,
        registered_model_name=DURATION_MODEL_NAME,
    )
    dur_run_id = mlflow.active_run().info.run_id

# Store feature column order for inference alignment
duration_feature_cols = list(X_train.columns)

avg_duration_overall = float(y_train.mean())
avg_duration_by_hour = y_train.groupby(X_train["hour_of_day"]).mean().to_dict()
print(f"  Avg duration (all): {avg_duration_overall:.1f} min")

print("Trip Duration Model — Enriched HistGradientBoosting")
print("=" * 50)
print(f"  RMSE:  {dur_metrics['rmse']:.2f} min")
print(f"  MAE:   {dur_metrics['mae']:.2f} min")
print(f"  R²:    {dur_metrics['r2']:.4f}")
print(f"\n  Features: {X_train.shape[1]} (base + weather + congestion + holidays)")
print(f"  Duration cap: ≤{DURATION_CAP_MIN} min")
print(f"  MLflow run ID: {dur_run_id}")
print(f"  Registered as: {DURATION_MODEL_NAME}")

## Agent Tools

Five tools that the LLM can call during a conversation. Every feature passed to the model at inference was also present during training.

| Tool | Data source | Features for model | Notes |
|------|-------------|-------------------|-------|
| `get_datetime_features` | System clock | hour_of_day, day_of_week, is_weekend, is_holiday | Current NYC time |
| `geocode_address` | Nominatim (OSM) | pickup/dropoff lat/lon → zone bins + haversine distance | Free, no API key |
| `get_weather` | Open-Meteo forecast | temperature_f, precipitation_inch, snowfall_inch, wind_speed_mph | Same features as training (historical archive) |
| `get_congestion` | Historical avg from training data | hourly_trip_count | Lookup by (hour, day_of_week) |
| `predict_duration` | Trained HistGBT model | All above combined | Returns predicted minutes + road distance |

In [0]:
import requests
import pytz
from datetime import datetime

# Spark dayofweek convention: 1=Sun, 2=Mon, ..., 7=Sat
# Python weekday():           0=Mon, 1=Tue, ..., 6=Sun
_PYTHON_TO_SPARK_DOW = {0: 2, 1: 3, 2: 4, 3: 5, 4: 6, 5: 7, 6: 1}


@tool
def get_datetime_features() -> dict:
    """Get the current NYC date and time as model features (hour_of_day, day_of_week, is_weekend, is_holiday). Always call this first."""
    now = datetime.now(pytz.timezone("America/New_York"))
    python_dow = now.weekday()
    date_str = now.strftime("%Y-%m-%d")
    return {
        "hour_of_day": now.hour,
        "day_of_week": _PYTHON_TO_SPARK_DOW[python_dow],
        "is_weekend": 1 if python_dow >= 5 else 0,
        "is_holiday": 1 if date_str in HOLIDAYS else 0,
        "day_name": now.strftime("%A"),
        "current_time": now.strftime("%Y-%m-%d %H:%M %Z"),
    }


@tool
def geocode_address(address: str) -> dict:
    """Convert an NYC street address or landmark name to latitude/longitude coordinates.
    Args:
        address: Street address or NYC landmark (e.g. 'Times Square', '34th St Penn Station', 'JFK Airport')
    """
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": f"{address}, New York City", "format": "json", "limit": 1},
        headers={"User-Agent": "nyc-taxi-duration-agent/1.0"},
        timeout=10,
    )
    resp.raise_for_status()
    results = resp.json()
    if not results:
        return {
            "error": f"Could not find location: {address!r}. Please ask the user to clarify or provide a more specific NYC address."
        }
    return {
        "lat": float(results[0]["lat"]),
        "lon": float(results[0]["lon"]),
        "display_name": results[0]["display_name"],
    }


@tool
def get_weather(lat: float, lon: float) -> dict:
    """Get current weather conditions (temperature, precipitation, snowfall, wind). These are features the ML model was trained on.
    Args:
        lat: Latitude
        lon: Longitude
    """
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,precipitation,snowfall,wind_speed_10m",
            "temperature_unit": "fahrenheit",
            "wind_speed_unit": "mph",
            "precipitation_unit": "inch",
            "timezone": "America/New_York",
        },
        timeout=10,
    )
    resp.raise_for_status()
    c = resp.json()["current"]
    return {
        "temperature_f": c["temperature_2m"],
        "precipitation_inch": c["precipitation"],
        "snowfall_inch": c["snowfall"],
        "wind_speed_mph": c["wind_speed_10m"],
    }


@tool
def get_congestion(hour_of_day: int, day_of_week: int) -> dict:
    """Get the historical average taxi demand (trip count) for this hour and day of week. This is a congestion proxy the ML model was trained on.
    Args:
        hour_of_day: Hour (0-23)
        day_of_week: Spark convention: 1=Sun, 2=Mon, ..., 7=Sat
    """
    key = (hour_of_day, day_of_week)
    avg_trips = congestion_lookup.get(key, congestion_median)
    return {"hourly_trip_count": int(avg_trips)}


@tool
def predict_duration(
    pickup_lat: float,
    pickup_lon: float,
    dropoff_lat: float,
    dropoff_lon: float,
    hour_of_day: int,
    day_of_week: int,
    is_weekend: int,
    is_holiday: int,
    temperature_f: float,
    precipitation_inch: float,
    snowfall_inch: float,
    wind_speed_mph: float,
    hourly_trip_count: int,
    passenger_count: int = 1,
    rate_code_id: int = 1,
) -> dict:
    """Predict NYC taxi trip duration using the trained ML model. You MUST pass ALL features: location coords, time, weather, congestion, and holiday flag. Call this last after collecting all inputs from the other tools.
    Args:
        pickup_lat: Pickup latitude
        pickup_lon: Pickup longitude
        dropoff_lat: Dropoff latitude
        dropoff_lon: Dropoff longitude
        hour_of_day: Current hour (0-23)
        day_of_week: 1=Sun, 2=Mon, ..., 7=Sat
        is_weekend: 1 if Sat/Sun, else 0
        is_holiday: 1 if US federal holiday, else 0
        temperature_f: Current temperature in Fahrenheit
        precipitation_inch: Current precipitation in inches
        snowfall_inch: Current snowfall in inches
        wind_speed_mph: Current wind speed in mph
        hourly_trip_count: Historical avg taxi trips for this hour/day
        passenger_count: Number of passengers (default 1)
        rate_code_id: TLC rate code: 1=Standard, 2=JFK, 3=Newark (default 1)
    """
    bin_size = LAT_LON_BIN_SIZE

    pickup_lat_bin = round(pickup_lat / bin_size) * bin_size
    pickup_lon_bin = round(pickup_lon / bin_size) * bin_size
    dropoff_lat_bin = round(dropoff_lat / bin_size) * bin_size
    dropoff_lon_bin = round(dropoff_lon / bin_size) * bin_size

    # Haversine distance (miles) × 1.3 road-distance correction
    R = 3958.8
    lat1, lon1 = math.radians(pickup_lat), math.radians(pickup_lon)
    lat2, lon2 = math.radians(dropoff_lat), math.radians(dropoff_lon)
    a = (
        math.sin((lat2 - lat1) / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    )
    trip_distance = round(2 * R * math.asin(math.sqrt(a)) * 1.3, 2)

    # One-hot encode rate_code_id
    rc_cols = {f"rc_{i}": (1 if rate_code_id == i else 0) for i in range(1, 7)}

    row = {
        "hour_of_day": hour_of_day,
        "day_of_week": day_of_week,
        "is_weekend": is_weekend,
        "trip_distance": trip_distance,
        "passenger_count": passenger_count,
        "temperature_f": temperature_f,
        "precipitation_inch": precipitation_inch,
        "snowfall_inch": snowfall_inch,
        "wind_speed_mph": wind_speed_mph,
        "hourly_trip_count": hourly_trip_count,
        "is_holiday": is_holiday,
        "pickup_lat_bin": pickup_lat_bin,
        "pickup_lon_bin": pickup_lon_bin,
        "dropoff_lat_bin": dropoff_lat_bin,
        "dropoff_lon_bin": dropoff_lon_bin,
        **rc_cols,
    }
    features = pd.DataFrame([row])

    # Align to training column order; fill any unseen OHE columns with 0
    for col in duration_feature_cols:
        if col not in features.columns:
            features[col] = 0
    features = features[duration_feature_cols]

    predicted_min = max(1.0, float(duration_model.predict(features)[0]))

    # Baseline: same route under average/neutral conditions
    baseline_row = {
        **row,
        "temperature_f": avg_weather["temperature_f"],
        "precipitation_inch": avg_weather["precipitation_inch"],
        "snowfall_inch": avg_weather["snowfall_inch"],
        "wind_speed_mph": avg_weather["wind_speed_mph"],
        "hourly_trip_count": congestion_median,
        "is_holiday": 0,
    }
    baseline_features = pd.DataFrame([baseline_row])
    for col in duration_feature_cols:
        if col not in baseline_features.columns:
            baseline_features[col] = 0
    baseline_features = baseline_features[duration_feature_cols]
    baseline_min = max(1.0, float(duration_model.predict(baseline_features)[0]))

    pct_vs_typical = round((predicted_min - baseline_min) / baseline_min * 100, 1)
    return {
        "predicted_duration_min": round(predicted_min, 1),
        "typical_duration_this_route_min": round(baseline_min, 1),
        "pct_vs_typical": pct_vs_typical,
        "estimated_road_distance_miles": trip_distance,
    }


print(
    "Tools defined: get_datetime_features, geocode_address, get_weather, "
    "get_congestion, predict_duration"
)

## Trip Duration Agent

The agent uses Databricks Foundation Model API (OpenAI-compatible) with tool/function calling.  
The LLM orchestrates all tool calls autonomously — the user just asks in plain English.

In [0]:
_TOOLS = [
    get_datetime_features,
    geocode_address,
    get_weather,
    get_congestion,
    predict_duration,
]

_SYSTEM_PROMPT = """You are a helpful NYC taxi trip duration assistant.

When a user asks how long a taxi journey will take, you MUST follow these steps IN ORDER:
1. Call get_datetime_features() to get the current time, day, and holiday status.
2. Call geocode_address() for the pickup location.
3. Call geocode_address() for the dropoff location.
4. Call get_weather() at the pickup coordinates.
5. Call get_congestion() with the hour_of_day and day_of_week from step 1.
6. Call predict_duration() passing ALL collected data: coordinates, time features,
   weather features, congestion, and holiday flag.
7. Respond with the estimated duration, road distance, and a brief note on conditions
   (weather, congestion level, time of day, holiday status).

IMPORTANT:
- The ML model was trained on weather, congestion, and holiday data alongside taxi trip data.
  Every feature must be passed to predict_duration — do NOT skip any.
- Always use the tools — never guess coordinates, distances, or durations.
- If a location is ambiguous, ask the user to clarify before calling tools.

The format of your response should be 1. A VERY short sentence saying the predicted time and whether current conditions make it longer or shorter than typical for this route (with the percentage), and 2. In another paragraph, a very brief description of the conditions (weather, congestion, time of day) and how they affect this specific trip."""

# ── Build LangGraph agent ────────────────────────────────────────────────────
_system_message = {"role": "system", "content": _SYSTEM_PROMPT}
llm_with_tools = llm.bind_tools(_TOOLS)


def _routing_logic(state):
    last = state["messages"][-1]
    if last.get("tool_calls"):
        return "use_tool"
    return "end"


_preprocessor = RunnableLambda(lambda state: [_system_message] + state["messages"])
_model_runnable = _preprocessor | llm_with_tools


def _call_model(state, config):
    response = _model_runnable.invoke(state, config)
    return {"messages": [response]}


_workflow = StateGraph(ChatAgentState)
_workflow.add_node("agent", RunnableLambda(_call_model))
_workflow.add_node("tools", ChatAgentToolNode(_TOOLS))
_workflow.set_entry_point("agent")
_workflow.add_conditional_edges(
    "agent", _routing_logic, {"use_tool": "tools", "end": END}
)
_workflow.add_edge("tools", "agent")

trip_agent = _workflow.compile()


def ask_agent(user_input: str) -> str:
    """Run the trip duration agent with a natural-language query."""
    result = trip_agent.invoke({"messages": [{"role": "user", "content": user_input}]})
    return result["messages"][-1]["content"]


print("Agent ready (LangGraph).")
print(
    "Call: ask_agent('I am at Times Square and want a taxi to JFK. How long will it take?')"
)

## Demo

Change the query below and re-run the cell (Shift+Enter) to try different trips.

In [0]:
# ── Quick Start (1/2): install packages & restart ─────────────────────────────
# After this cell finishes, run the next cell to load the agent.
%pip install mlflow scikit-learn requests pytz langchain langgraph databricks-langchain --quiet
%restart_python

In [0]:
# ── Quick Start (2/2): load model + build agent (no retraining) ───────────────
# Takes ~1-2 min. Loads model from MLflow registry, builds lookups, defines tools.
# This cell is standalone — run after Quick Start (1/2) to skip retraining.

import importlib
import math  # noqa: F811
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import pytz  # noqa: F811
import requests  # noqa: F811
import src.constants
import src.transforms
from databricks_langchain import ChatDatabricks
from datetime import datetime  # noqa: F811
from langchain_core.runnables import RunnableLambda
from langchain_core.tools import tool
from langgraph.graph import END, StateGraph
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from pyspark.sql import functions as F

importlib.reload(src.constants)
from src.constants import (  # noqa: E402, F811
    AGENT_LLM_ENDPOINT,
    DURATION_MODEL_NAME,
    HOLIDAYS,
    LAT_LON_BIN_SIZE,
    SILVER_TABLE,
    WEATHER_TABLE,
)

mlflow.langchain.autolog()

# ── 1) Load model from registry ──────────────────────────────────────────────
print("Loading model from MLflow registry...")
_client = mlflow.tracking.MlflowClient()
_versions = _client.search_model_versions(f"name='{DURATION_MODEL_NAME}'")
_latest_v = max(int(v.version) for v in _versions)
duration_model = mlflow.sklearn.load_model(f"models:/{DURATION_MODEL_NAME}/{_latest_v}")
duration_feature_cols = list(duration_model.feature_names_in_)
print(f"  Model loaded (v{_latest_v}): {len(duration_feature_cols)} features")

# ── 2) Congestion lookup ─────────────────────────────────────────────────────
print("Building congestion lookup...")
silver_df = spark.read.table(SILVER_TABLE)
silver_df = silver_df.withColumn(
    "trip_date", F.to_date("tpep_pickup_datetime")
).withColumn("trip_hour", F.hour("tpep_pickup_datetime"))
_cdf = silver_df.groupBy("trip_date", "trip_hour").agg(
    F.count("*").alias("hourly_trip_count")
)
_lk = (
    _cdf.withColumn("dow", F.dayofweek("trip_date"))
    .groupBy(F.col("trip_hour").alias("lk_hour"), F.col("dow").alias("lk_dow"))
    .agg(F.round(F.avg("hourly_trip_count")).cast("int").alias("avg_trips"))
    .toPandas()
)
congestion_lookup = dict(zip(zip(_lk["lk_hour"], _lk["lk_dow"]), _lk["avg_trips"]))
congestion_median = int(np.median(list(congestion_lookup.values())))
print(f"  Congestion: {len(congestion_lookup)} entries, median {congestion_median:,}")

# ── 3) Average weather baseline ──────────────────────────────────────────────
_wpdf = spark.read.table(WEATHER_TABLE).toPandas()
avg_weather = {
    "temperature_f": float(_wpdf["temperature_f"].mean()),
    "precipitation_inch": float(_wpdf["precipitation_inch"].mean()),
    "snowfall_inch": float(_wpdf["snowfall_inch"].mean()),
    "wind_speed_mph": float(_wpdf["wind_speed_mph"].mean()),
}

# ── 4) Define tools ──────────────────────────────────────────────────────────
_PYTHON_TO_SPARK_DOW = {0: 2, 1: 3, 2: 4, 3: 5, 4: 6, 5: 7, 6: 1}


@tool
def get_datetime_features() -> dict:
    """Get the current NYC date and time as model features (hour_of_day, day_of_week, is_weekend, is_holiday). Always call this first."""
    now = datetime.now(pytz.timezone("America/New_York"))
    dow = now.weekday()
    return {
        "hour_of_day": now.hour,
        "day_of_week": _PYTHON_TO_SPARK_DOW[dow],
        "is_weekend": 1 if dow >= 5 else 0,
        "is_holiday": 1 if now.strftime("%Y-%m-%d") in HOLIDAYS else 0,
        "day_name": now.strftime("%A"),
        "current_time": now.strftime("%Y-%m-%d %H:%M %Z"),
    }


@tool
def geocode_address(address: str) -> dict:
    """Convert an NYC street address or landmark name to lat/lon coordinates.
    Args:
        address: Street address or NYC landmark (e.g. 'Times Square', 'JFK Airport')
    """
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": f"{address}, New York City", "format": "json", "limit": 1},
        headers={"User-Agent": "nyc-taxi-duration-agent/1.0"},
        timeout=10,
    )
    resp.raise_for_status()
    results = resp.json()
    if not results:
        return {
            "error": f"Could not find location: {address!r}. Please ask the user to clarify."
        }
    return {
        "lat": float(results[0]["lat"]),
        "lon": float(results[0]["lon"]),
        "display_name": results[0]["display_name"],
    }


@tool
def get_weather(lat: float, lon: float) -> dict:
    """Get current weather conditions (temperature, precipitation, snowfall, wind).
    Args:
        lat: Latitude
        lon: Longitude
    """
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,precipitation,snowfall,wind_speed_10m",
            "temperature_unit": "fahrenheit",
            "wind_speed_unit": "mph",
            "precipitation_unit": "inch",
            "timezone": "America/New_York",
        },
        timeout=10,
    )
    resp.raise_for_status()
    c = resp.json()["current"]
    return {
        "temperature_f": c["temperature_2m"],
        "precipitation_inch": c["precipitation"],
        "snowfall_inch": c["snowfall"],
        "wind_speed_mph": c["wind_speed_10m"],
    }


@tool
def get_congestion(hour_of_day: int, day_of_week: int) -> dict:
    """Get historical average taxi demand for this hour and day of week.
    Args:
        hour_of_day: Hour (0-23)
        day_of_week: 1=Sun, 2=Mon, ..., 7=Sat
    """
    return {
        "hourly_trip_count": int(
            congestion_lookup.get((hour_of_day, day_of_week), congestion_median)
        )
    }


@tool
def predict_duration(
    pickup_lat: float,
    pickup_lon: float,
    dropoff_lat: float,
    dropoff_lon: float,
    hour_of_day: int,
    day_of_week: int,
    is_weekend: int,
    is_holiday: int,
    temperature_f: float,
    precipitation_inch: float,
    snowfall_inch: float,
    wind_speed_mph: float,
    hourly_trip_count: int,
    passenger_count: int = 1,
    rate_code_id: int = 1,
) -> dict:
    """Predict NYC taxi trip duration. Pass ALL features from the other tools.
    Args:
        pickup_lat: Pickup latitude
        pickup_lon: Pickup longitude
        dropoff_lat: Dropoff latitude
        dropoff_lon: Dropoff longitude
        hour_of_day: Current hour (0-23)
        day_of_week: 1=Sun..7=Sat
        is_weekend: 1 if Sat/Sun else 0
        is_holiday: 1 if holiday else 0
        temperature_f: Temperature in F
        precipitation_inch: Precipitation in inches
        snowfall_inch: Snowfall in inches
        wind_speed_mph: Wind speed in mph
        hourly_trip_count: Avg taxi trips for this hour/day
        passenger_count: Passengers (default 1)
        rate_code_id: TLC rate code (default 1)
    """
    bs = LAT_LON_BIN_SIZE
    plb, plob = round(pickup_lat / bs) * bs, round(pickup_lon / bs) * bs
    dlb, dlob = round(dropoff_lat / bs) * bs, round(dropoff_lon / bs) * bs
    R = 3958.8
    la1, lo1, la2, lo2 = (
        math.radians(pickup_lat),
        math.radians(pickup_lon),
        math.radians(dropoff_lat),
        math.radians(dropoff_lon),
    )
    a = (
        math.sin((la2 - la1) / 2) ** 2
        + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    )
    trip_distance = round(2 * R * math.asin(math.sqrt(a)) * 1.3, 2)
    rc = {f"rc_{i}": (1 if rate_code_id == i else 0) for i in range(1, 7)}
    row = {
        "hour_of_day": hour_of_day,
        "day_of_week": day_of_week,
        "is_weekend": is_weekend,
        "trip_distance": trip_distance,
        "passenger_count": passenger_count,
        "temperature_f": temperature_f,
        "precipitation_inch": precipitation_inch,
        "snowfall_inch": snowfall_inch,
        "wind_speed_mph": wind_speed_mph,
        "hourly_trip_count": hourly_trip_count,
        "is_holiday": is_holiday,
        "pickup_lat_bin": plb,
        "pickup_lon_bin": plob,
        "dropoff_lat_bin": dlb,
        "dropoff_lon_bin": dlob,
        **rc,
    }
    feat = pd.DataFrame([row])
    for c in duration_feature_cols:
        if c not in feat.columns:
            feat[c] = 0
    feat = feat[duration_feature_cols]
    pred = max(1.0, float(duration_model.predict(feat)[0]))
    brow = {
        **row,
        "temperature_f": avg_weather["temperature_f"],
        "precipitation_inch": avg_weather["precipitation_inch"],
        "snowfall_inch": avg_weather["snowfall_inch"],
        "wind_speed_mph": avg_weather["wind_speed_mph"],
        "hourly_trip_count": congestion_median,
        "is_holiday": 0,
    }
    bf = pd.DataFrame([brow])
    for c in duration_feature_cols:
        if c not in bf.columns:
            bf[c] = 0
    bf = bf[duration_feature_cols]
    base = max(1.0, float(duration_model.predict(bf)[0]))
    pct = round((pred - base) / base * 100, 1)
    return {
        "predicted_duration_min": round(pred, 1),
        "typical_duration_this_route_min": round(base, 1),
        "pct_vs_typical": pct,
        "estimated_road_distance_miles": trip_distance,
    }


# ── 5) Build LangGraph agent ─────────────────────────────────────────────────
_TOOLS = [
    get_datetime_features,
    geocode_address,
    get_weather,
    get_congestion,
    predict_duration,
]
_SYSTEM_PROMPT = """You are a helpful NYC taxi trip duration assistant.

When a user asks how long a taxi journey will take, you MUST follow these steps IN ORDER:
1. Call get_datetime_features() to get the current time, day, and holiday status.
2. Call geocode_address() for the pickup location.
3. Call geocode_address() for the dropoff location.
4. Call get_weather() at the pickup coordinates.
5. Call get_congestion() with the hour_of_day and day_of_week from step 1.
6. Call predict_duration() passing ALL collected data.
7. Respond with the estimated duration and a brief note on conditions.

IMPORTANT:
- Always use the tools — never guess coordinates, distances, or durations.
- If a location is ambiguous or not found, ask the user to clarify.

The format of your response should be 1. A VERY short sentence saying the predicted time and whether current conditions make it longer or shorter than typical for this route (with the percentage), You only need to reply with the predicted duration and typical duration without the preamble, and 2. In another paragraph, a very brief, but honest/down-to-earth/human sounding description of the conditions (weather, congestion, time of day) and how they affect this specific trip."""

_smsg = {"role": "system", "content": _SYSTEM_PROMPT}
llm = ChatDatabricks(endpoint=AGENT_LLM_ENDPOINT)
_llm_t = llm.bind_tools(_TOOLS)
_pre = RunnableLambda(lambda s: [_smsg] + s["messages"])
_mr = _pre | _llm_t


def _call_model(state, config):
    return {"messages": [_mr.invoke(state, config)]}


def _route(state):
    return "use_tool" if state["messages"][-1].get("tool_calls") else "end"


_wf = StateGraph(ChatAgentState)
_wf.add_node("agent", RunnableLambda(_call_model))
_wf.add_node("tools", ChatAgentToolNode(_TOOLS))
_wf.set_entry_point("agent")
_wf.add_conditional_edges("agent", _route, {"use_tool": "tools", "end": END})
_wf.add_edge("tools", "agent")
trip_agent = _wf.compile()


def ask_agent(user_input: str) -> str:  # noqa: F811
    result = trip_agent.invoke({"messages": [{"role": "user", "content": user_input}]})
    return result["messages"][-1]["content"]


print("✅ Agent ready. Run a demo cell below.")

In [0]:
query = "Times Square to JFK."
print(f"User: {query}\n")
answer = ask_agent(query)
print(f"\nAgent:\n{answer}")